# M0 — `maml_inner_steps_eval` Sweep (Val Set)

Post-hoc sweep over adaptation step counts at test time.  
The model is **loaded once** from a checkpoint, then re-evaluated on the **validation set** for each step count.  
No training happens here.

**Config source:** best trial from the v4 warm-start HPO (trial 0: `outer_lr=1.009e-4`, `maml_inner_steps=10`, etc.).  
Architecture dims come from `ablation_hpo.py` fixed constants: `cnn_base_filters=64`, `lstm_hidden=64`, `groupnorm_num_groups=8`.

> ⚠️ **Before running:** set `CHECKPOINT_PATH`, `CODE_DIR`, and `DATA_DIR` in the *Paths* cell.

## 0. Paths — **edit these**

In [4]:
import os

# ── SET THESE ────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = r"C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\M0_trial64_best.pt"   # <-- your saved .pt file
CODE_DIR        = r"C:\Users\kdmen\Repos\pers-gest-cls"
DATA_DIR        = r"C:\Users\kdmen\Repos\pers-gest-cls\dataset"
# C:\Users\kdmen\Repos\pers-gest-cls\dataset\meta-learning-sup-que-ds\segfilt_rts_tensor_dict.pkl
# ─────────────────────────────────────────────────────────────────────────────

os.environ["CODE_DIR"] = CODE_DIR
os.environ["DATA_DIR"] = DATA_DIR
os.environ["RUN_DIR"]  = "/tmp/inner_steps_sweep"   # scratch; nothing important saved here

## 1. Imports & sys.path

In [5]:
import sys
import copy
import json
import pickle
import warnings

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pathlib import Path

warnings.filterwarnings("ignore", message=".*weights_only=False.*")

_code = Path(CODE_DIR)
for _p in [
    _code,
    _code / "system",
    _code / "system" / "MAML",
    _code / "system" / "MOE",
    _code / "system" / "pretraining",
]:
    sys.path.insert(0, str(_p))

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.7.1+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 4070 Laptop GPU


## 2. Config

Built from `ablation_config.make_base_config("M0")`, then overwritten with the **best v4 HPO trial** (trial 0 from the warm-start list, highest-ranked trial in the document you provided).

**Architecture dims** are the fixed constants from `ablation_hpo.py`:  
`cnn_base_filters=64`, `lstm_hidden=64`, `groupnorm_num_groups=8`.

**Notes on the chosen trial:**
- `maml_alpha_init_eval = 0.00672` — on the lower end across the top-10. This means each gradient step moves cautiously; the model may need more steps to converge at test time than a trial with a larger eval LR would. Worth watching — if the curve is still rising at 100 steps you may want to re-sweep with a larger `maml_alpha_init_eval`.
- `use_lslr_at_eval = False` — per-layer step-size scaling is **off** at eval. The single `maml_alpha_init_eval` scalar drives all layers. This makes the inner_steps sensitivity more pronounced.
- `MOE_aux_loss_plcmt = 'outer'` — aux loss only in the outer loop. Fine for eval (aux loss is not used during adaptation anyway).
- `maml_msl_num_epochs = 39` (`use_maml_msl = 'hybrid'`) — multi-step loss was on during training. This is compatible with any eval step count.

In [ ]:
from ablation_config import make_base_config

config = make_base_config(ablation_id="M0")

# ── Architecture dims: fixed constants from ablation_hpo.py ──────────────────
config["cnn_base_filters"]     = 64
config["lstm_hidden"]          = 64
config["groupnorm_num_groups"] = 8
config["use_GlobalAvgPooling"] = True   # set in build_config_from_trial in hpo

# ── Best HPO trial (v4 warm-start, trial 0) ───────────────────────────────────
# Source: M0_WARM_START_PARAMS[0] from ablation_hpo.py
config["learning_rate"]          = 0.00010093304999603776   # outer_lr
config["weight_decay"]           = 0.0008325426470137959
config["maml_inner_steps"]       = 10
config["maml_alpha_init"]        = 0.0009888781900907544
config["maml_alpha_init_eval"]   = 0.006717556958813548
config["maml_use_lslr"]          = True
config["use_lslr_at_eval"]       = False
config["use_maml_msl"]           = "hybrid"
config["maml_msl_num_epochs"]    = 39
config["episodes_per_epoch_train"] = 250
config["num_experts"]            = 24
config["MOE_top_k"]              = 7
config["top_k"]                  = 7
config["MOE_gate_temperature"]   = 1.1879664247660187
config["MOE_aux_coeff"]          = 0.08672942143224953
config["MOE_ctx_out_dim"]        = 64
config["MOE_ctx_hidden_dim"]     = 32
config["MOE_dropout"]            = 0.022501513050004283
config["apply_MOE_aux_loss_inner_outer"] = "outer"   # MOE_aux_loss_plcmt
config["label_smooth"]           = 0.05    # fixed for MAML in HPO

# ── Fixed MAML settings (from _suggest_maml_hps) ────────────────────────────
config["meta_learning"]          = True
config["meta_batchsize"]         = 24
config["maml_opt_order"]         = "first"
config["maml_first_order_to_second_order_epoch"] = 1_000_000
config["enable_inner_loop_optimizable_bn_params"] = False

# ── Fixed MoE settings (from _suggest_moe_hps) ───────────────────────────────
config["use_MOE"]              = True
config["MOE_placement"]        = "encoder"
config["MOE_expert_expand"]    = 1.0
config["MOE_mlp_hidden_mult"]  = 1.0
config["MOE_log_every"]        = 5
config["MOE_plot_dir"]         = None
config["gate_type"]            = "context_feature_demo"
config["expert_architecture"]  = "MLP"

# ── Eval settings ─────────────────────────────────────────────────────────────
config["num_eval_episodes"]  = 200   # NUM_VAL_EPISODES from ablation_config
config["padding"]            = 0
config["use_batch_norm"]     = False

print("Config built. Key values:")
for k in ["cnn_base_filters", "lstm_hidden", "groupnorm_num_groups",
          "maml_inner_steps", "maml_alpha_init_eval", "use_lslr_at_eval",
          "num_experts", "MOE_top_k", "use_MOE"]:
    print(f"  {k}: {config[k]}")

CODE_DIR: C:\Users\kdmen\Repos\pers-gest-cls
DATA_DIR: C:\Users\kdmen\Repos\pers-gest-cls\dataset
RUN_DIR : C:\tmp\inner_steps_sweep
Using fold 0: 24 train | 4 val | 4 test PIDs
Config built. Key values:
  cnn_base_filters: 64
  lstm_hidden: 64
  groupnorm_num_groups: 8
  maml_inner_steps: 10
  maml_alpha_init_eval: 0.006717556958813548
  use_lslr_at_eval: False
  num_experts: 24
  MOE_top_k: 7
  use_MOE: True


## 3. Build model & load checkpoint

In [10]:
from ablation_config import build_maml_moe_model, count_parameters

model = build_maml_moe_model(config)
print(f"Parameters: {count_parameters(model):,}")

from MAML.mamlpp import PerParamPerStepLSLR, named_param_dict

# ── Recreate _lslr on the model before loading checkpoint ─────────────────────
# _lslr is monkey-patched onto the model during training (not part of the
# architecture), so we must recreate it here before load_state_dict will accept
# the checkpoint's keys.
if config["maml_use_lslr"]:
    temp_params = named_param_dict(model, require_grad_only=True)
    model._lslr = PerParamPerStepLSLR(
        named_params = temp_params.items(),
        inner_steps  = config["maml_inner_steps"],   # must match what was used at train time
        init_lr      = config["maml_alpha_init"],
        learnable    = True,
        device       = config["device"],
    ).to(config["device"])

Parameters: 6,003,451


In [11]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=config["device"])


In [12]:
# The checkpoint saved by M0_full_model.py stores state_dict under 'model_state_dict'.
# If your checkpoint has a different structure, adjust accordingly.
state_dict = ckpt["model_state_dict"]
model.load_state_dict(state_dict)
model.eval()

print(f"Checkpoint loaded from: {CHECKPOINT_PATH}")
if "best_val_acc" in ckpt:
    print(f"Checkpoint best_val_acc: {ckpt['best_val_acc']:.4f}")
if "seed" in ckpt:
    print(f"Checkpoint seed: {ckpt['seed']}")

Checkpoint loaded from: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\M0_trial64_best.pt


## 4. Sweep

In [13]:
from MAML.maml_data_pipeline import MetaGestureDataset, maml_mm_collate, reorient_tensor_dict
from MAML.mamlpp import mamlpp_adapt_and_eval
from torch.utils.data import DataLoader
from collections import defaultdict
from ablation_config import VAL_PIDS, FIXED_SEED

INNER_STEPS_SWEEP = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 75, 100]
NUM_VAL_EPISODES  = 200

tensor_dict_path = os.path.join(config["dfs_load_path"], "segfilt_rts_tensor_dict.pkl")

with open(tensor_dict_path, "rb") as f:
    full_dict = pickle.load(f)
tensor_dict = reorient_tensor_dict(full_dict, config)

val_ds = MetaGestureDataset(
    tensor_dict,
    target_pids             = VAL_PIDS,
    target_gesture_classes  = config["maml_gesture_classes"],
    target_trial_indices    = config["target_trial_indices"],
    n_way                   = config["n_way"],
    k_shot                  = config["k_shot"],
    q_query                 = config["q_query"],
    num_eval_episodes       = NUM_VAL_EPISODES,
    is_train                = False,
    seed                    = FIXED_SEED,
    use_label_shuf_meta_aug = False,
)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=False,
                    num_workers=4, collate_fn=maml_mm_collate)

print(f"Val episodes: {len(val_ds)}")
print(f"Val PIDs    : {VAL_PIDS}")

Val episodes: 800
Val PIDs    : ['P011', 'P006', 'P105', 'P109']


In [14]:
sweep_results = []   # list of dicts: {steps, mean_acc, std_acc, mean_loss, std_loss, per_user}

for n_steps in INNER_STEPS_SWEEP:
    sweep_config = copy.deepcopy(config)
    sweep_config["maml_inner_steps_eval"] = n_steps

    model.eval()
    user_accs  = defaultdict(list)
    user_losses = defaultdict(list)

    for batch in val_dl:
        uid     = batch["user_id"]
        metrics = mamlpp_adapt_and_eval(
            model, sweep_config, batch["support"], batch["query"]
        )
        user_accs[uid].append(metrics["acc"])
        # mamlpp_adapt_and_eval may or may not return query_loss depending on your version.
        # Capture it if available, otherwise skip loss tracking.
        if "query_loss" in metrics:
            user_losses[uid].append(metrics["query_loss"])

    per_user_acc  = {uid: float(np.mean(accs))   for uid, accs   in user_accs.items()}
    per_user_loss = {uid: float(np.mean(losses)) for uid, losses in user_losses.items()} if user_losses else {}

    all_accs   = list(per_user_acc.values())
    mean_acc   = float(np.mean(all_accs))
    std_acc    = float(np.std(all_accs))

    if per_user_loss:
        all_losses = list(per_user_loss.values())
        mean_loss  = float(np.mean(all_losses))
        std_loss   = float(np.std(all_losses))
    else:
        mean_loss = std_loss = float("nan")

    sweep_results.append({
        "inner_steps_eval": n_steps,
        "mean_acc":  mean_acc,
        "std_acc":   std_acc,
        "mean_loss": mean_loss,
        "std_loss":  std_loss,
        "per_user_acc":  per_user_acc,
        "per_user_loss": per_user_loss,
    })

    print(f"steps={n_steps:3d}  val_acc={mean_acc*100:.2f}% ± {std_acc*100:.2f}%"
          + (f"  val_loss={mean_loss:.4f} ± {std_loss:.4f}" if not np.isnan(mean_loss) else ""))

steps=  5  val_acc=82.12% ± 2.64%


KeyboardInterrupt: 

## 5. Results table

In [ ]:
print(f"{'inner_steps_eval':>18}  {'mean_acc (%)':>13}  {'std_acc (%)':>12}  {'mean_loss':>10}")
print("-" * 62)
for r in sweep_results:
    loss_str = f"{r['mean_loss']:.4f}" if not np.isnan(r['mean_loss']) else "   N/A"
    print(f"{r['inner_steps_eval']:>18}  {r['mean_acc']*100:>13.2f}  {r['std_acc']*100:>12.2f}  {loss_str:>10}")

## 6. Plots

In [ ]:
steps  = [r["inner_steps_eval"] for r in sweep_results]
accs   = [r["mean_acc"] * 100   for r in sweep_results]
stds   = [r["std_acc"]  * 100   for r in sweep_results]
losses = [r["mean_loss"]        for r in sweep_results]
has_loss = not np.isnan(losses[0])

n_plots = 2 if has_loss else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(steps, accs, marker="o", linewidth=2, color="steelblue", label="mean val acc")
ax.fill_between(steps,
                [a - s for a, s in zip(accs, stds)],
                [a + s for a, s in zip(accs, stds)],
                alpha=0.2, color="steelblue", label="±1 std (over users)")
best_idx = int(np.argmax(accs))
ax.axvline(steps[best_idx], color="red", linestyle="--", alpha=0.6,
           label=f"best: {steps[best_idx]} steps ({accs[best_idx]:.2f}%)")

# Mark train-time step count
train_steps = config["maml_inner_steps"]
ax.axvline(train_steps, color="orange", linestyle=":", alpha=0.8,
           label=f"train steps = {train_steps}")

ax.set_xlabel("maml_inner_steps_eval", fontsize=12)
ax.set_ylabel("Val Accuracy (%)", fontsize=12)
ax.set_title("M0 — Inner Steps Eval Sweep (Val Set)", fontsize=13)
ax.xaxis.set_major_locator(mticker.FixedLocator(steps))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Loss (if available) ───────────────────────────────────────────────────────
if has_loss:
    ax2 = axes[1]
    loss_stds = [r["std_loss"] for r in sweep_results]
    ax2.plot(steps, losses, marker="s", linewidth=2, color="coral", label="mean val loss")
    ax2.fill_between(steps,
                     [l - s for l, s in zip(losses, loss_stds)],
                     [l + s for l, s in zip(losses, loss_stds)],
                     alpha=0.2, color="coral")
    ax2.axvline(train_steps, color="orange", linestyle=":", alpha=0.8,
                label=f"train steps = {train_steps}")
    ax2.set_xlabel("maml_inner_steps_eval", fontsize=12)
    ax2.set_ylabel("Val Loss", fontsize=12)
    ax2.set_title("M0 — Query Loss vs Eval Steps", fontsize=13)
    ax2.xaxis.set_major_locator(mticker.FixedLocator(steps))
    ax2.tick_params(axis='x', rotation=45)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("M0_inner_steps_eval_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nBest: {steps[best_idx]} steps → {accs[best_idx]:.2f}% ± {stds[best_idx]:.2f}%")

## 7. Per-user breakdown at best step count

In [ ]:
best_result = sweep_results[best_idx]
per_user = best_result["per_user_acc"]

uids   = sorted(per_user.keys())
u_accs = [per_user[uid] * 100 for uid in uids]

fig, ax = plt.subplots(figsize=(max(8, len(uids) * 0.6), 4))
ax.bar(range(len(uids)), u_accs, color="steelblue", alpha=0.8)
ax.axhline(np.mean(u_accs), color="red", linestyle="--",
           label=f"mean = {np.mean(u_accs):.2f}%")
ax.axhline(100 / config["n_way"], color="gray", linestyle=":",
           label=f"chance = {100/config['n_way']:.1f}%")
ax.set_xticks(range(len(uids)))
ax.set_xticklabels(uids, rotation=60, fontsize=8)
ax.set_ylabel("Val Accuracy (%)", fontsize=12)
ax.set_title(f"Per-user accuracy @ {steps[best_idx]} inner steps", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("M0_per_user_best_steps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
assert(False)

## 8. Save results JSON

In [ ]:
import json

# per_user dicts have int/float values already, but uid keys may be ints — convert
serialisable = []
for r in sweep_results:
    entry = {k: v for k, v in r.items() if k not in ("per_user_acc", "per_user_loss")}
    entry["per_user_acc"]  = {str(k): v for k, v in r["per_user_acc"].items()}
    entry["per_user_loss"] = {str(k): v for k, v in r["per_user_loss"].items()}
    serialisable.append(entry)

out = {
    "ablation_id":         "M0",
    "sweep_type":          "maml_inner_steps_eval",
    "val_pids":            VAL_PIDS,
    "num_val_episodes":    NUM_VAL_EPISODES,
    "maml_inner_steps_train": config["maml_inner_steps"],
    "maml_alpha_init_eval":   config["maml_alpha_init_eval"],
    "use_lslr_at_eval":       config["use_lslr_at_eval"],
    "checkpoint":          CHECKPOINT_PATH,
    "results":             serialisable,
    "best_steps":          steps[best_idx],
    "best_mean_acc":       accs[best_idx] / 100,
}

with open("M0_inner_steps_eval_sweep.json", "w") as f:
    json.dump(out, f, indent=2)

print("Saved: M0_inner_steps_eval_sweep.json")